**AI Support Resolution Agent - End-to-End Capstone Project**

**PHASE 1 - UNDERSTAND THE PROBLEM & DEFINE SUCCESS**


**Industry: E-commerce / SaaS Customer Support**

**AI Agent Name: SupportGenie AI**

**Primary User Persona: Tier-1 Customer Support Executive**

**Daily workflow:**

Responds to 100+ customer tickets/day
Handles refund/order/account questions
Needs quick access to company policies
Must escalate fraud/legal/sensitive cases
Cannot expose customer PII
Works under SLA pressure

**Exact Problem to Solve**

Customer support agents spend excessive time:

Searching documentation,
Responding inconsistently,
Escalating incorrectly,
Manually handling repetitive queries.

The AI agent should:

Answer common support questions,
Retrieve accurate policy information,
Use tools to fetch order/ticket details,
Escalate unsafe/sensitive requests,
Preserve conversation context,
Avoid hallucinations.

**Inputs**
Customer query
Chat history
Policy documents
Order/ticket metadata
Feedback signals

**Outputs**
Support response
Resolution recommendation
Escalation decision
Tool outputs
Safety refusal if needed

**Constraints**
No hallucinated policies
No PII storage in logs
Must escalate fraud/legal threats
Must refuse policy-violating requests
Must fail gracefully


**Assumptions**
Internal policy documents exist
API tools are mocked locally
OpenAI API key available
Small vector database acceptable

**Example User Questions**
“Where is my order?”
“Can I get a refund after 45 days?”
“Delete another user’s account.”
“I forgot my password.”
“Why was my payment declined?”

**Success Criteria**
    
Policy accuracy	>90%
Hallucination rate	<5%
Unsafe response rate	0%
Retrieval relevance	>85%
Tool routing accuracy	>90%
Escalation accuracy	>95%


**Known Failure Cases**

Hallucinated refund policy -> Legal risk
Wrong tool usage ->	Incorrect resolution
Infinite tool loop -> Runtime failure
Missing retrieval context -> Wrong answers
PII leakage in logs	-> Compliance issue
Adversarial prompt injection ->	Unsafe behaviour

**Architecture Overview**

User Query
   ↓
Safety Guardrails
   ↓
Planner / Router
   ↓
RAG Retriever
   ↓
Tool Selection Layer
   ↓
LLM Reasoning
   ↓
Memory + Context
   ↓
Response Generator
   ↓
Feedback Storage

**Recommended Tech Stack**

Framework --> 	LangChain
LLM	--> GPT-4 / GPT-4o-mini
Embeddings -->	OpenAIEmbeddings
Vector DB -->	Chroma
Backend	--> Python
Logging	-> LangSmith / Python logging
Deployment -->	Streamlit / FastAPI
Memory --> ConversationBufferMemory
Tools -->	LangChain Tools

**PHASE 2 - BUILD A BASIC WORKING AGENT**

The goal of Phase 2 is to intentionally build a simple but limited baseline system without LLM. This baseline agent should:

Accept customer input
Perform simple rule-based classification
Generate static responses
Handle invalid inputs
Log interactions
Apply basic safety checks
Demonstrate clear limitations

The below code demonstrates a CLI based system that applies keyword based rules & policies with input & sanity checks along with logging and exception handling. 

In [0]:
import logging
from datetime import datetime
import re

# -----------------------------
# LOGGING CONFIGURATION
# -----------------------------

logging.basicConfig(
    filename="support_agent.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# -----------------------------
# SAMPLE COMPANY POLICIES
# -----------------------------

REFUND_POLICY = """
Refunds are allowed within 30 days of purchase
for unused products with proof of purchase.
"""

SHIPPING_POLICY = """
Orders are usually delivered within 5-7 business days.
"""

ESCALATION_MESSAGE = """
Your request has been escalated to a human support agent.
"""

# -----------------------------
# SAFETY RULES
# -----------------------------

UNSAFE_KEYWORDS = [
    "hack",
    "delete another user",
    "steal",
    "bypass payment",
    "fake refund"
]

SENSITIVE_KEYWORDS = [
    "legal",
    "lawsuit",
    "sue",
    "fraud",
    "police"
]

# -----------------------------
# INPUT VALIDATION
# -----------------------------

def validate_input(user_input):
    """
    Validate customer query.
    """

    if not user_input.strip():
        return False, "Input cannot be empty."

    if len(user_input) > 500:
        return False, "Input too long."

    return True, None

# -----------------------------
# PII REDACTION FOR LOGGING
# -----------------------------

def sanitize_for_logging(text):
    """
    Remove emails and phone numbers from logs.
    """

    text = re.sub(r'\S+@\S+', '[EMAIL_REDACTED]', text)

    text = re.sub(r'\b\d{10}\b', '[PHONE_REDACTED]', text)

    return text

# -----------------------------
# BASIC RULE-BASED AGENT
# -----------------------------

def support_agent(user_query):

    query = user_query.lower()

    # -------------------------
    # SAFETY FILTER
    # -------------------------

    for word in UNSAFE_KEYWORDS:
        if word in query:

            response = (
                "I cannot assist with unsafe or unauthorized requests."
            )

            return response

    # -------------------------
    # SENSITIVE CASE ESCALATION
    # -------------------------

    for word in SENSITIVE_KEYWORDS:
        if word in query:

            response = ESCALATION_MESSAGE

            return response

    # -------------------------
    # ORDER STATUS
    # -------------------------

    if "where is my order" in query:
        return "Please provide your order ID."

    # -------------------------
    # REFUND QUESTIONS
    # -------------------------

    elif "refund" in query:
        return REFUND_POLICY

    # -------------------------
    # SHIPPING QUESTIONS
    # -------------------------

    elif "shipping" in query or "delivery" in query:
        return SHIPPING_POLICY

    # -------------------------
    # PASSWORD RESET
    # -------------------------

    elif "password" in query:
        return (
            "You can reset your password using the "
            "'Forgot Password' link on the login page."
        )

    # -------------------------
    # ACCOUNT DELETION
    # -------------------------

    elif "delete my account" in query:
        return (
            "Account deletion requests require identity verification. "
            "A support ticket has been created."
        )

    # -------------------------
    # DEFAULT FALLBACK
    # -------------------------

    else:
        return (
            "I'm unable to answer that question. "
            "Your request has been escalated."
        )

# -----------------------------
# INTERACTION LOGGER
# -----------------------------

def log_interaction(user_input, response):

    sanitized_input = sanitize_for_logging(user_input)

    sanitized_response = sanitize_for_logging(response)

    logging.info(
        f"USER: {sanitized_input} | BOT: {sanitized_response}"
    )

# -----------------------------
# MAIN CLI LOOP
# -----------------------------

def run_agent():

    print("=" * 50)
    print("SupportGenie AI - Baseline Rule-Based Agent")
    print("Type 'exit' to quit.")
    print("=" * 50)

    while True:

        user_input = input("\nCustomer: ")

        # Exit condition
        if user_input.lower() == "exit":
            print("Session ended.")
            break

        # Validate input
        valid, error = validate_input(user_input)

        if not valid:
            print(f"Agent: {error}")
            continue

        try:

            # Generate response
            response = support_agent(user_input)

            # Print response
            print(f"Agent: {response}")

            # Log interaction
            log_interaction(user_input, response)

        except Exception as e:

            error_message = (
                "Temporary issue detected. "
                "Please contact support."
            )

            print(f"Agent: {error_message}")

            logging.error(str(e))

# -----------------------------
# RUN APPLICATION
# -----------------------------

if __name__ == "__main__":
    run_agent()

SupportGenie AI - Baseline Rule-Based Agent
Type 'exit' to quit.
Agent: You can reset your password using the 'Forgot Password' link on the login page.
Agent: 
Refunds are allowed within 30 days of purchase
for unused products with proof of purchase.

Agent: I cannot assist with unsafe or unauthorized requests.
Agent: 
Your request has been escalated to a human support agent.

Agent: I'm unable to answer that question. Your request has been escalated.
Agent: I'm unable to answer that question. Your request has been escalated.
Agent: Please provide your order ID.
Agent: I'm unable to answer that question. Your request has been escalated.
Agent: I'm unable to answer that question. Your request has been escalated.
Agent: I'm unable to answer that question. Your request has been escalated.
Agent: I'm unable to answer that question. Your request has been escalated.
Agent: 
Refunds are allowed within 30 days of purchase
for unused products with proof of purchase.

Agent: 
Orders are usually 

**Sample outputs and limitations of rule based agent:**


Customer: Can I get a refund?

Agent: Refunds are allowed within 30 days of purchase
for unused products with proof of purchase.

Customer: I want to hack another account

Agent: I cannot assist with unsafe or unauthorized requests.

Customer: Where is my order?

Agent: Please provide your order ID.


**Limitation 1: No contexual understanding**

Customer: I bought headphones last year and they stopped working.
Can I return them?

Agent: Refunds are allowed within 30 days.

Problems with this: 

No reasoning
No warranty understanding
No damaged-product handling
No contextual interpretation


**Limitation 2: No policy grounding**

Cusotmer: Do you support refunds for international crypto payments?
Agent: Refunds are allowed within 30 days.

Problems:

Agent always provides a generic response with no policy retrieval and verification, resulting in high risk for misinformation. 


**Limitation 3: No multi turn memory**

Customer: Where is my order?
Agent: Please provide your order ID.
Customer: 1234
Agent: I'm unable to answer that question.


Problems: There is no conversational state & memory maintained. 


PHASE 3 - MAKE THE AGENT SMARTER

In [19]:
import os
import logging
import re
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

# -----------------------------------
# LOAD ENV VARIABLES
# -----------------------------------

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# -----------------------------------
# LOGGING CONFIGURATION
# -----------------------------------

logging.basicConfig(
    filename="logs/phase3_agent.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# -----------------------------------
# INITIALIZE LLM
# -----------------------------------

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3,
    max_tokens=300
)

# -----------------------------------
# SAFETY KEYWORDS
# -----------------------------------

UNSAFE_KEYWORDS = [
    "hack",
    "steal account",
    "delete another user",
    "fake refund",
    "bypass payment"
]

SENSITIVE_KEYWORDS = [
    "lawsuit",
    "legal",
    "fraud",
    "police",
    "threat"
]

# -----------------------------------
# PII SANITIZATION
# -----------------------------------

def sanitize_text(text):

    text = re.sub(r'\S+@\S+', '[EMAIL_REDACTED]', text)

    text = re.sub(r'\b\d{10}\b', '[PHONE_REDACTED]', text)

    return text

# -----------------------------------
# INPUT VALIDATION
# -----------------------------------

def validate_input(query):

    if not query.strip():
        return False, "Input cannot be empty."

    if len(query) > 1000:
        return False, "Input too long."

    return True, None

# -----------------------------------
# SAFETY PRECHECK
# -----------------------------------

def safety_check(query):

    q = query.lower()

    for word in UNSAFE_KEYWORDS:
        if word in q:
            return (
                False,
                "I cannot assist with unauthorized or unsafe requests."
            )

    for word in SENSITIVE_KEYWORDS:
        if word in q:
            return (
                False,
                "This case has been escalated to a human support agent."
            )

    return True, None


# -----------------------------------
# PROMPT V1 - BASIC
# -----------------------------------

prompt_v1 = PromptTemplate(
    input_variables=["query"],
    template="""
You are a customer support assistant.

Answer the customer query professionally.

Customer Query:
{query}
"""
)


# -----------------------------------
# PROMPT V2 - SAFETY FOCUSED
# -----------------------------------

prompt_v2 = PromptTemplate(
    input_variables=["query"],
    template="""
You are a safe customer support AI assistant.

Rules:
- Never invent company policies
- Never provide unsafe instructions
- Refuse unauthorized requests
- Escalate legal or fraud-related issues
- If unsure, say you do not know

Customer Query:
{query}
"""
)


# -----------------------------------
# PROMPT V3 - STRUCTURED + ESCALATION
# -----------------------------------

prompt_v3 = PromptTemplate(
    input_variables=["query"],
    template="""
You are SupportGenie AI.

Your responsibilities:
1. Help customers professionally
2. Never hallucinate policies
3. Escalate unresolved or sensitive cases
4. Refuse unsafe requests
5. Keep responses concise

Response format:
- Intent
- Action Taken
- Final Response

If policy information is unavailable,
explicitly say so and escalate.

Customer Query:
{query}
"""
)

# -----------------------------------
# CHAIN EXECUTOR
# -----------------------------------

def run_chain(prompt, query):

    chain = LLMChain(
        llm=llm,
        prompt=prompt
    )

    response = chain.run(query=query)

    return response

# -----------------------------------
# RESPONSE LOGGER
# -----------------------------------

def log_interaction(query, response, version):

    safe_query = sanitize_text(query)

    safe_response = sanitize_text(response)

    logging.info(
        f"PROMPT_VERSION={version} | "
        f"QUERY={safe_query} | "
        f"RESPONSE={safe_response}"
    )


# -----------------------------------
# MAIN AGENT EXECUTION
# -----------------------------------

def run_agent():

    print("=" * 60)
    print("SupportGenie AI - Phase 3 LLM Agent")
    print("=" * 60)

    while True:

        user_query = input("\nCustomer: ")

        if user_query.lower() == "exit":
            break

        # -------------------------
        # INPUT VALIDATION
        # -------------------------

        valid, error = validate_input(user_query)

        if not valid:
            print(f"\nAgent: {error}")
            continue

        # -------------------------
        # SAFETY PRECHECK
        # -------------------------

        safe, safety_response = safety_check(user_query)

        if not safe:
            print(f"\nAgent: {safety_response}")

            log_interaction(
                user_query,
                safety_response,
                "SAFETY_BLOCK"
            )

            continue

        # -------------------------
        # RUN ALL PROMPT VARIANTS
        # -------------------------

        try:

            print("\n--- PROMPT V1 OUTPUT ---")
            response_v1 = run_chain(prompt_v1, user_query)
            print(response_v1)

            print("\n--- PROMPT V2 OUTPUT ---")
            response_v2 = run_chain(prompt_v2, user_query)
            print(response_v2)

            print("\n--- PROMPT V3 OUTPUT ---")
            response_v3 = run_chain(prompt_v3, user_query)
            print(response_v3)

            # ---------------------
            # LOGGING
            # ---------------------

            log_interaction(
                user_query,
                response_v1,
                "V1"
            )

            log_interaction(
                user_query,
                response_v2,
                "V2"
            )

            log_interaction(
                user_query,
                response_v3,
                "V3"
            )

        except Exception as e:

            logging.error(str(e))

            print(
                "\nAgent: Temporary issue detected. "
                "Please try again later."
            )

# -----------------------------------
# APPLICATION ENTRY
# -----------------------------------

if __name__ == "__main__":
    run_agent()

SupportGenie AI - Phase 3 LLM Agent


**Prompt comparison analysis**

**QUERY 1:**

What is the status of my order?

--- PROMPT V1 OUTPUT ---

Dear Customer,

Thank you for reaching out to us regarding your order. To assist you better, could you please provide your order number or any additional details related to your purchase? This will help us track your order and provide you with the most accurate information.

We appreciate your patience and look forward to resolving this for you.

Best regards,  
[Your Name]  
Customer Support Team

--- PROMPT V2 OUTPUT ---
I can help you with that! However, I will need some more information to assist you. Could you please provide your order number or the email address associated with your order?

--- PROMPT V3 OUTPUT ---
- Intent: Order Status Inquiry
- Action Taken: Unable to provide specific order information; escalating the case.
- Final Response: I can't access your order details. Please contact customer support directly for assistance.

Analysis:

Prompt 1 and 2 asked for the order number while Prompt 3 escalated, which is quite unnecessary.

**QUERY 2:**

Can I get a refund after 41 days?

--- PROMPT V1 OUTPUT ---
Thank you for reaching out. Our refund policy typically allows for refunds within a specific timeframe, which is usually outlined at the time of purchase. If it has been 41 days since your transaction, it may be outside of our standard refund window. However, I recommend contacting our customer service team directly with your order details, as they may be able to assist you further or provide alternative solutions. Please let me know if you need any additional information!

--- PROMPT V2 OUTPUT ---
I’m sorry, but I cannot provide specific information about refund policies. Typically, refund timelines can vary by company and product. I recommend checking the company's official refund policy or contacting customer support directly for assistance with your request.

--- PROMPT V3 OUTPUT ---
- Intent: Refund Inquiry
- Action Taken: Unable to provide specific policy information; escalating the case.
- Final Response: I cannot provide specific policy details regarding refunds after 41 days. I will escalate your inquiry for further assistance.


Analysis:

Prompt 1 - vague and not able to answer accurately
Prompt 2 - correct answer 
Prompt 3 - correct answer, structured output and analysis, escalation for further assitance


**QUERY 3:**

Can I delete Jim's account ?


--- PROMPT V1 OUTPUT ---
Dear Customer,

Thank you for reaching out. To delete Jim's account, please follow these steps:

1. Log in to your account.
2. Navigate to the "Account Settings" or "Manage Accounts" section.
3. Locate Jim's account in the list of users.
4. Select the option to delete or deactivate the account.
5. Follow any additional prompts to confirm the deletion.

If you encounter any issues or need further assistance, please don't hesitate to contact us.

Best regards,  
[Your Name]  
Customer Support Team

--- PROMPT V2 OUTPUT ---
I'm sorry, but I can't assist with that. If you need to delete an account, please contact the appropriate customer support team or follow the official procedures outlined by the company.

--- PROMPT V3 OUTPUT ---
- Intent: Account Deletion Inquiry
- Action Taken: Unable to provide specific policy information; escalating the case.
- Final Response: I cannot provide information on deleting Jim's account. I will escalate this matter for further assistance.

Agent: I cannot assist with unauthorized or unsafe requests.


Analysis:

Prompt 1 - Wrong answer! Cannot provide assitance to delete another user's account. Legal risk. 
Prompt 2 - Partially correct and vague answer. 
Prompt 3 - Identified that this is an unauthorized or unsafe request. 

**QUERY 4:**

I want to sue you. 


--- PROMPT V1 OUTPUT ---
Dear [Customer's Name],

Thank you for reaching out to us. I’m sorry to hear that you are feeling this way. We take customer concerns very seriously and would like to understand the issue you are facing.

Please provide us with more details about your situation, and we will do our best to address your concerns promptly. Our goal is to resolve any issues amicably and to your satisfaction.

Thank you for your patience, and we look forward to hearing from you soon.

Best regards,

[Your Name]  
Customer Support Team  
[Your Company]  
[Contact Information]  

--- PROMPT V2 OUTPUT ---
I'm sorry to hear that you're feeling this way. If you have a specific issue or concern, I recommend reaching out to our customer support team directly to discuss it further. They will be able to assist you with your situation.

--- PROMPT V3 OUTPUT ---
- Intent: Customer expressing intent to take legal action
- Action Taken: Acknowledge the concern and escalate the issue
- Final Response: I understand your frustration. I recommend contacting our legal department directly for assistance with your concerns.

Analysis:

Prompt 1: Empathetic but vague answer
Prompt 2: Correct answer
Prompt 3: Correct analysis, structured output and escalated to the right department (legal)


**QUERY 5:**

Do you offer refunds in cryptocurrency?

--- PROMPT V1 OUTPUT ---
Thank you for reaching out to us. Currently, we do not offer refunds in cryptocurrency. Our refund policy is limited to traditional payment methods. If you have any further questions or need assistance with a specific transaction, please let us know, and we’ll be happy to help!

--- PROMPT V2 OUTPUT ---
I’m sorry, but I cannot provide information about company policies regarding refunds, including those related to cryptocurrency. I recommend checking our official website or contacting customer support directly for accurate information.

--- PROMPT V3 OUTPUT ---
- Intent: Inquiry about refund policy for cryptocurrency
- Action Taken: Policy information unavailable; escalating the case for further assistance.
- Final Response: I currently do not have information on our policy regarding crypto refunds. I will escalate this case for further assistance.

Analysis:

Prompt1: Vague but correct answer
Prompt2: Checked policies and gave right recommendation, hallucinating risk
Prompt 3: Checked policy and escalated 

**Prompt Comparison Table:**

Metric	                 V1 Basic	      V2 Safety	       V3 Structured
Naturalness	             High	          Medium	        Medium
Safety	                 Low	          High	            High
Hallucination Risk	     High	          Medium	        Low
Escalation Handling	     Weak	          Medium	        Strong
Explainability	         Weak	          Medium	        Strong
Operational Readiness	 Low	          Medium	        High


**Summary:**

Prompt 3 is recommended since it provides a structured answer, has less hallucination risk, checks against policies & safety violations, provides explanation for decisions it takes, escalates correctly. 

**Improvement areas:**

Risk of false policy creations and unnecessary escalations. 
Need for more grounding through stronger prompts and better escalation rules.



**PHASE 4 - ADD KNOWLEDGE RETREIVAL**

In this phase, the customer support bot will be enhanced with RAG to:

1. Answer using verified company policies,
2. Reduce hallucinations,
3. Provide explainable answers,
4. Escalate only when information is unavailable.

Key objectives of this phase:

1. Prepare documents for embedding
2. Implement embeddings
3. Semantic search
4. Connect retrieval to responses
5. Compare with/without retrieval
6. Missing info handling
7. Failure analysis
8. Retrieval evaluation

**Architecture**

User Query
   ↓
Input Validation
   ↓
Safety Layer
   ↓
Embedding-Based Retrieval
   ↓
Top Relevant Chunks
   ↓
LLM + Retrieved Context
   ↓
Grounded Response
   ↓
Escalation / Fallback

**RAG Workflow**

1. Load company documents
2. Split into chunks
3. Convert chunks into embeddings
4. Store embeddings in vector DB
5. Embed user query
6. Perform semantic similarity search
7. Retrieve top-k chunks
8. Inject retrieved context into prompt
9. Generate grounded response
10. Escalate if confidence is low

**Tech Stack**

| Component  | Technology                     |
| ---------- | ------------------------------ |
| Framework  | LangChain                      |
| Embeddings | OpenAIEmbeddings               |
| Vector DB  | Chroma                         |
| Chunking   | RecursiveCharacterTextSplitter |
| Retriever  | Similarity Search              |
| Backend    | Python                         |


In [2]:
!pip install chromadb
!pip install tiktoken

Defaulting to user installation because normal site-packages is not writeableRequirement already satisfied: chromadb in /usr/local/lib/python3.10/site-packages (0.4.18)Requirement already satisfied: requests>=2.28 in /usr/local/lib/python3.10/site-packages (from chromadb) (2.32.5)Requirement already satisfied: pydantic>=1.9 in /usr/local/lib/python3.10/site-packages (from chromadb) (2.11.7)Requirement already satisfied: chroma-hnswlib==0.7.3 in /usr/local/lib/python3.10/site-packages (from chromadb) (0.7.3)Requirement already satisfied: fastapi>=0.95.2 in /usr/local/lib/python3.10/site-packages (from chromadb) (0.116.1)Requirement already satisfied: uvicorn>=0.18.3 in /usr/local/lib/python3.10/site-packages (from uvicorn[standard]>=0.18.3->chromadb) (0.24.0.post1)Requirement already satisfied: posthog>=2.4.0 in /usr/local/lib/python3.10/site-packages (from chromadb) (3.0.2)Requirement already satisfied: typing-extensions>=4.5.0 in /usr/local/lib/python3.10/site-packages (from chromadb)

In [12]:
#TEXT LOADER 

from langchain.document_loaders import TextLoader
import os

documents = []

data_folder = "data"

for file in os.listdir(data_folder):

    if file.endswith(".txt"):

        loader = TextLoader(
            os.path.join(data_folder, file)
        )

        documents.extend(loader.load())

In [13]:
#CHUNKING
#Chunking strategy - Given the smaller size of the policy documents, had to finetune to get a #meaningful chunk size of 70 to get the full context with a bit of overlap to get continuity. 

from langchain.text_splitter import (
    RecursiveCharacterTextSplitter
)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=70,
    chunk_overlap=20
)

splits = text_splitter.split_documents(documents)

In [14]:
__import__('pysqlite3')

import sys

sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [15]:
#Embeddings

from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

In [14]:
#Vector DB - Chroma Implementation

#from langchain.vectorstores import Chroma

#vectordb = Chroma.from_documents(
#   documents=splits,
#    embedding=embeddings,
#    persist_directory="./vector_db"
#)

AttributeError: module 'chromadb' has no attribute 'config'

In [16]:
import sqlite3

print(sqlite3.sqlite_version)

3.45.0


In [17]:
#Using fiass instead of chroma db since its compatible with vocarium

!pip install faiss-cpu

Defaulting to user installation because normal site-packages is not writeableRequirement already satisfied: faiss-cpu in /usr/local/lib/python3.10/site-packages (1.7.4)

In [17]:
from langchain.vectorstores import FAISS

vectordb = FAISS.from_documents(
    splits,
    embeddings
)

In [18]:
#Create retreiver to get top 3 (k=3) chunks

retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [19]:
#Test retreival

results = retriever.get_relevant_documents(
    "Can I get a refund after 45 days?"
)

for doc in results:
    print(doc.page_content)


#From the output i see that chunk size is too high. Fine tuning it to get the right context. See good results (as below output) with chunk size of 70 and chunk overlap of 20. 

/tmp/ipykernel_80/992229116.py:3: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  results = retriever.get_relevant_documents(


Customers may request refunds within 30 days of purchase.
Refunds after 30 days require managerial approval.
Refund Policy


In [20]:
#Test retreival

results = retriever.get_relevant_documents(
    "My package is late"
)

for doc in results:
    print(doc.page_content)


#The chunk size works for this query too. Testing chunk size for multiple queries. 


if delivery exceeds 20 business days.
Shipping Policy

Standard delivery time:
5-7 business days.
- unused products
- damaged products
- incorrect deliveries


In [25]:
#Connect RAG retreival to prompt template

from langchain.prompts import PromptTemplate

rag_prompt = PromptTemplate(
    input_variables=["context", "query"],
    template="""
You are SupportGenie AI.

Use ONLY the provided context.

If the answer is unavailable:
- say you do not know 
- be very professional
- escalate to support

Context:
{context}

Customer Query:
{query}

Response:
"""
)

In [26]:
#Create RAG Chain

from langchain.chains import LLMChain
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.2
)

rag_chain = LLMChain(
    llm=llm,
    prompt=rag_prompt
)

In [27]:
#Run agent with full RAG execution

def rag_support_agent(user_query):

    # --------------------------------
    # RETRIEVE RELEVANT DOCUMENTS
    # --------------------------------

    retrieved_docs = retriever.get_relevant_documents(
        user_query
    )

    # --------------------------------
    # BUILD CONTEXT
    # --------------------------------

    context = "\n\n".join(
        [doc.page_content for doc in retrieved_docs]
    )

    # --------------------------------
    # HANDLE EMPTY RETRIEVAL
    # --------------------------------

    if not context.strip():

        return (
            "I could not find verified policy "
            "information. Escalating to support."
        )

    # --------------------------------
    # GENERATE RESPONSE
    # --------------------------------

    response = rag_chain.run(
        context=context,
        query=user_query
    )

    return response

In [28]:
import logging
import re

# ----------------------------------------
# LOGGING CONFIGURATION
# ----------------------------------------

logging.basicConfig(
    filename="logs/rag_agent.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

# ----------------------------------------
# SAFETY RULES
# ----------------------------------------

UNSAFE_KEYWORDS = [
    "hack",
    "delete another user",
    "steal account",
    "fake refund",
    "bypass payment"
]

SENSITIVE_KEYWORDS = [
    "lawsuit",
    "legal",
    "fraud",
    "police",
    "harassment"
]

# ----------------------------------------
# INPUT VALIDATION
# ----------------------------------------

def validate_input(query):

    if not query.strip():
        return False, "Input cannot be empty."

    if len(query) > 1000:
        return False, "Input too long."

    return True, None

# ----------------------------------------
# SAFETY CHECK
# ----------------------------------------

def safety_check(query):

    q = query.lower()

    # Unsafe requests
    for word in UNSAFE_KEYWORDS:

        if word in q:

            return (
                False,
                "I cannot assist with unsafe or unauthorized requests."
            )

    # Sensitive escalation
    for word in SENSITIVE_KEYWORDS:

        if word in q:

            return (
                False,
                "This request has been escalated to a human support agent."
            )

    return True, None

# ----------------------------------------
# PII SANITIZATION
# ----------------------------------------

def sanitize_for_logging(text):

    # Remove emails
    text = re.sub(
        r'\S+@\S+',
        '[EMAIL_REDACTED]',
        text
    )

    # Remove phone numbers
    text = re.sub(
        r'\b\d{10}\b',
        '[PHONE_REDACTED]',
        text
    )

    return text

# ----------------------------------------
# LOGGING FUNCTION
# ----------------------------------------

def log_interaction(query, response):

    safe_query = sanitize_for_logging(query)

    safe_response = sanitize_for_logging(response)

    logging.info(
        f"QUERY={safe_query} | RESPONSE={safe_response}"
    )

# ----------------------------------------
# MAIN APPLICATION LOOP
# ----------------------------------------

def run_rag_agent():

    print("=" * 60)
    print("SupportGenie AI - Phase 4 RAG Agent")
    print("Type 'exit' to quit")
    print("=" * 60)

    while True:

        # --------------------------------
        # GET USER INPUT
        # --------------------------------

        user_query = input("\nCustomer: ")

        # --------------------------------
        # EXIT CONDITION
        # --------------------------------

        if user_query.lower() == "exit":

            print("\nSession ended.")

            break

        # --------------------------------
        # INPUT VALIDATION
        # --------------------------------

        valid, error = validate_input(user_query)

        if not valid:

            print(f"\nAgent: {error}")

            continue

        # --------------------------------
        # SAFETY CHECK
        # --------------------------------

        safe, safety_response = safety_check(user_query)

        if not safe:

            print(f"\nAgent: {safety_response}")

            log_interaction(
                user_query,
                safety_response
            )

            continue

        # --------------------------------
        # CALL RAG AGENT
        # --------------------------------

        try:

            response = rag_support_agent(user_query)

            print(f"\nAgent: {response}")

            log_interaction(
                user_query,
                response
            )

        # --------------------------------
        # ERROR HANDLING
        # --------------------------------

        except Exception as e:

            logging.error(str(e))

            error_message = (
                "Temporary system issue detected. "
                "Please try again later."
            )

            print(f"\nAgent: {error_message}")

# ----------------------------------------
# APPLICATION ENTRY POINT
# ----------------------------------------

if __name__ == "__main__":

    run_rag_agent()

SupportGenie AI - Phase 4 RAG Agent
Type 'exit' to quit

Agent: I do not know. However, refunds after 30 days require managerial approval. I recommend reaching out to our support team for further assistance with your request.

Agent: Yes, you can request a refund within 15 days of your purchase. Please let us know how we can assist you with the process.

Agent: I cannot assist with unsafe or unauthorized requests.

Agent: Thank you for reaching out. I understand your concern regarding the delay in your delivery. According to our shipping policy, standard delivery time is typically 5-7 business days. If your delivery exceeds 20 business days, please let us know, and we can assist you further. 

If you have any additional questions or require further assistance, I recommend contacting our support team for a more detailed resolution. Thank you for your patience.

Agent: Thank you for reaching out. I understand that you would like to request a refund. According to our refund policy, custom

**Output analysis in comparison to without RAG implementation**

**QUERY 1: Can I refund after 45 days?**

Agent: I do not know. However, refunds after 30 days require managerial approval. I recommend reaching out to our support team for further assistance with your request.

Analysis: Agent answered based on refund policy and prompt. Fine tuned prompt to make a professional response to get the above response. 

Can I refund after 15 days?

Agent: Yes, you can request a refund within 15 days of your purchase. Please let us know how we can assist you with the process.

Analysis : This is as per the policy document. 

**QUERY 2: Can i delete another user's account?**

Agent: I cannot assist with unsafe or unauthorized requests.

Analsyis: This is as per the account_security policy document. The answer is correct and accurate. 

**QUERY 3: Why is my delivery late?**

Agent: Thank you for reaching out. I understand your concern regarding the delay in your delivery. According to our shipping policy, standard delivery time is typically 5-7 business days. If your delivery exceeds 20 business days, please let us know, and we can assist you further. 

If you have any additional questions or require further assistance, I recommend contacting our support team for a more detailed resolution. Thank you for your patience.

Analysis: Answer is accurate as per the shipping policy and escalation policy. There is no immediate escalation but recommendation to contact the support team for more details. 

**QUERY 4: I want my payment back!**

Agent: Thank you for reaching out. I understand that you would like to request a refund. According to our refund policy, customers may request refunds within 30 days of purchase. Please provide me with your order details so I can assist you further. If you have any additional questions or concerns, I can escalate this to our support team for further assistance.

Analysis: This is accurate as per the escalation policy as well as the refund policy. 

**QUERY 5: Can you compensate for international shipping delays?**

Agent: I do not have information regarding compensation for international shipping delays. I recommend reaching out to our support team for further assistance on this matter. Thank you for your understanding.

Analysis: No information is available in any policy document on this. The "I do not know" in the answer reflects that. 

**QUERY 6: Do you accept refund in crypto currency?**

Agent: I do not know if cryptocurrency is an accepted payment method. I recommend reaching out to our support team for further assistance regarding payment options.

Analysis: Once again, information not available and answer reflects that. 

Session ended.

**EVALUATION - Fine tuning & handling cases where relevant information is missing**

1. Improved prompt - Reponses for queries not in policy where not handled correctly. Included this additional instruction in the prompt to reply with I do not know. But this response was very rude. Hence added additional instruction to respond professionally.

2. Chunk Sizes - Experimented with chunk size of 50, 100 and finally saw best responses at 70.

3. Hallucination Handling - Added explicit instruction in prompt to use ONLY the available content.

4. Escalation accuracy - I see that for some queries escalation is suggested but only those in the policy is escalated immediately. 


Evalaution Metrics:

| Metric              | Evidence                        |
| ------------------- | ------------------------------- |
| Retrieval Precision | Relevant chunks retrieved       |
| Groundedness        | Response matches retrieved docs |
| Hallucination Rate  | Fabricated policies             |
| Escalation Accuracy | Correct fallback behaviour      |
| Citation Accuracy   | Correct source references       |




**PHASE 5 - TOOLS ENABLEMENT**

This phase is required to demonstrate true agentic behaviour in terms of orchestration (selecting the right tool) via reasoning. This also demonstrates enterprise level integration with databases and other systems in real world environment.  

With this implementation, the agent can now:

Choose tools,
Execute functions,
Retrieve live information,
Perform operational tasks,
Make decisions dynamically.


The following tools will be implemented:

1. Order status checking tool - Takes an order id and checks if its in the database and returns the order status for that id.
2. Escalation tool - Function to a ticketing tool to handle escalation.
3. Refund Eligibility Tool - Check refund eligibility using a rule based function

| Tool             | Demonstrates          |
| ---------------- | --------------------- |
| Order Status     | External data access  |
| Escalation Tool  | Operational workflows |
| Refund Checker   | Business logic        |


Architecture:

Customer Query
      ↓
Safety Layer
      ↓
Intent Detection
      ↓
Tool Router
      ↓
Selected Tool
      ↓
Tool Execution
      ↓
Tool Result
      ↓
LLM Response Generator
      ↓
Logging


In [11]:
#Create Mock Database

ORDERS_DB = {
    "1001": {
        "status": "Shipped",
        "delivery_date": "2026-05-20"
    },
    "1002": {
        "status": "Processing",
        "delivery_date": "2026-05-25"
    },
    "1003": {
        "status": "Delivered",
        "delivery_date": "2026-05-10"
    }
}

In [12]:
#Tool 1 - Order status tool

def get_order_status(order_id):

    """
    Fetch order status from database.
    """

    if order_id in ORDERS_DB:

        order = ORDERS_DB[order_id]

        return (
            f"Order {order_id} is currently "
            f"{order['status']} "
            f"and expected delivery is "
            f"{order['delivery_date']}."
        )

    return "Order ID not found."

In [13]:
#Tool2 - Escalation tool

ESCALATION_QUEUE = []

def escalate_ticket(reason):

    """
    Escalate sensitive/unresolved issues.
    """

    ESCALATION_QUEUE.append(reason)

    return (
        "Your request has been escalated "
        "to a human support agent."
    )

In [14]:
#Tool3 - Refund eleigibility tool

def refund_eligibility(days_since_purchase):

    """
    Determine refund eligibility.
    """

    try:

        days = int(days_since_purchase)

        if days <= 30:

            return (
                "Customer is eligible for refund."
            )

        else:

            return (
                "Refund requires managerial approval."
            )

    except:

        return "Invalid refund duration."

In [15]:
#Convert functions into langchain tools

from langchain.agents import Tool

tools = [

    Tool(
        name="OrderStatusTool",

        func=get_order_status,

        description="""
        Use this tool when customers ask:
        - where is my order
        - track shipment
        - delivery status
        - shipping updates

        Input should ONLY be the order ID.
        """
    ),

    Tool(
        name="EscalationTool",

        func=escalate_ticket,

        description="""
        Use this tool for:
        - legal threats
        - fraud complaints
        - harassment
        - unresolved cases
        - sensitive customer issues

        Input should be escalation reason.
        """
    ),

    Tool(
        name="RefundEligibilityTool",

        func=refund_eligibility,

        description="""
        Use this tool when customer asks:
        - refund eligibility
        - return window
        - refund duration

        Input should ONLY be number of days.
        """
    )
]

In [18]:
from langchain.agents import initialize_agent
from langchain.agents import AgentType

agent = initialize_agent(

    tools=tools,

    llm=llm,

    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,

    verbose=True,

    max_iterations=3,

    handle_parsing_errors=True
)

NameError: name 'llm' is not defined

In [35]:
import logging

logging.basicConfig(
    filename="logs/tool_agent.log",
    level=logging.INFO
)

def run_tool_agent():

    print("=" * 60)
    print("SupportGenie AI - Tool Enabled Agent")
    print("=" * 60)

    while True:

        query = input("\nCustomer: ")

        if query.lower() == "exit":
            break

        try:

            response = agent.run(query)

            print(f"\nAgent: {response}")

            logging.info(
                f"QUERY={query} | RESPONSE={response}"
            )

        except Exception as e:

            logging.error(str(e))

            print(
                "\nAgent: Temporary issue detected."
            )

if __name__ == "__main__":

    run_tool_agent()

SupportGenie AI - Tool Enabled Agent


> Entering new AgentExecutor chain...
I need to track the shipment for order 1001 to provide the customer with the delivery status.  
Action: OrderStatusTool  
Action Input: 1001  
Observation: Order 1001 is currently Shipped and expected delivery is 2026-05-20.
Thought:I now know the final answer.  
Final Answer: Order 1001 is currently Shipped and is expected to be delivered on May 20, 2026.

> Finished chain.

Agent: Order 1001 is currently Shipped and is expected to be delivered on May 20, 2026.


> Entering new AgentExecutor chain...
I need to track the shipment for order 2000 to provide the customer with the delivery status.  
Action: OrderStatusTool  
Action Input: 2000  
Observation: Order ID not found.
Thought:The order ID 2000 is not found, which means I cannot provide a status update for this order. I should inform the customer about this issue.  
Final Answer: I'm sorry, but I couldn't find any information for order ID 2000. Please dou

**Analysis of tool execution**

Where is my order 1001?

Invoked OrderStatus tool as per the output:

> Entering new AgentExecutor chain...
I need to track the shipment for order 1001 to provide the customer with the delivery status.  
Action: OrderStatusTool  
Action Input: 1001  
Observation: Order 1001 is currently Shipped and expected delivery is 2026-05-20.
Thought:I now know the final answer.  
Final Answer: Order 1001 is currently Shipped and is expected to be delivered on May 20, 2026.

> Finished chain.

Agent: Order 1001 is currently Shipped and is expected to be delivered on May 20, 2026.


Where is order 2000?

Invoked OrderStatus tool and did not find the order in the database. Gave the right response as seen in the output: 

> Entering new AgentExecutor chain...
I need to track the shipment for order 2000 to provide the customer with the delivery status.  
Action: OrderStatusTool  
Action Input: 2000  
Observation: Order ID not found.
Thought:The order ID 2000 is not found, which means I cannot provide a status update for this order. I should inform the customer about this issue.  
Final Answer: I'm sorry, but I couldn't find any information for order ID 2000. Please double-check the order ID or contact customer support for further assistance.

> Finished chain.

Agent: I'm sorry, but I couldn't find any information for order ID 2000. Please double-check the order ID or contact customer support for further assistance.

Is a refund possible after 45 days?

Invoked the refund eligibility tool correctly and gave right response. 

> Entering new AgentExecutor chain...
I need to determine if a refund is possible after 45 days since the purchase. This requires checking the refund eligibility based on the number of days since the purchase.  
Action: RefundEligibilityTool  
Action Input: 45  
Observation: Refund requires managerial approval.
Thought:I now know that a refund after 45 days requires managerial approval, which suggests that it may not be automatically granted.  
Final Answer: A refund after 45 days requires managerial approval, and it may not be automatically granted.

> Finished chain.

Agent: A refund after 45 days requires managerial approval, and it may not be automatically granted.


I will sue you. 

Invoked escalation tool. 

> Entering new AgentExecutor chain...
This seems to be a serious issue that requires escalation. I need to use the EscalationTool to address this legal threat.  
Action: EscalationTool  
Action Input: legal threats  
Observation: Your request has been escalated to a human support agent.
Thought:I now know the final answer  
Final Answer: Your request has been escalated to a human support agent who will assist you with your legal concerns.

> Finished chain.

Agent: Your request has been escalated to a human support agent who will assist you with your legal concerns.

How do refunds work? 

Did not invoke order status tool but instead gave a generic answer, which is right behaviour.  

> Entering new AgentExecutor chain...
To answer the question about how refunds work, I need to provide general information about the refund process rather than using a specific tool. However, if the customer is asking about their specific refund eligibility or return window, I would need to know the number of days since the purchase to use the RefundEligibilityTool. Since the question is general, I will provide an overview instead.

Final Answer: Refunds typically involve returning a product within a specified period after purchase, and the amount refunded may depend on the condition of the item and the store's return policy. For specific details about eligibility and timelines, please refer to the store's refund policy or provide the number of days since purchase for a more tailored response.

> Finished chain.

Agent: Refunds typically involve returning a product within a specified period after purchase, and the amount refunded may depend on the condition of the item and the store's return policy. For specific details about eligibility and timelines, please refer to the store's refund policy or provide the number of days since purchase for a more tailored response.

In [0]:
#Tool failure - sometimes the question how does refund work might give a response saying order id not found because it selecets the OrderStatus tool. 

To safeguard against this, add some more prompts

prefix = """
You are a customer support routing agent.

Carefully choose tools.

Never use OrderStatusTool for:
- refunds
- billing
- cancellations
"""


#GUARDRAIL 1 - MAX ITERATIONS

Added the following guardrails:

To avoid loops, set the max iternation to 3 

#GUARDRAIL 2 - Intent recognition (call before tool execution)
if "refund" in query.lower():
    bypass_order_tool = True

#GUARDRAIL 3 - Validate the inputs passed to the tools. Ex: check if order id consists of digits

if not order_id.isdigit():
    return "Invalid order ID."





** PHASE 6 - PLANNING, MEMORY AND CONTEXT**

With memory, the agent can now remember context, demonstrate multi step reasoning and maintain coherent multi turn conversations. 

Objective is to add the below features and demonstrate conversation quality improvement:

1. Short term memory
2. Long term memory
3. Multi step reasoning & planning logic
4. Multi turn conversations
5. Memory rest rules
6. Safety aware memory 

| Concept                | Purpose                        |
| ---------------------- | ------------------------------ |
| Short-term memory      | Maintain session context       |
| Long-term memory       | Store persistent preferences   |
| Planning               | Break complex tasks into steps |
| Conversation state     | Multi-turn continuity          |
| Memory retention rules | Safety & privacy               |
| Context summarization  | Avoid token overflow           |


**Architecture:**

Customer Query
      ↓
Conversation Memory
      ↓
Planner / Reasoning Engine
      ↓
Tool Selection
      ↓
Retrieval Layer
      ↓
LLM Reasoning
      ↓
Response Generator
      ↓
Memory Update


In [17]:
#Use ConversationBufferMemory to store conversational dialogue and multi turn history

from langchain.memory import ConversationBufferMemory

memory = ConversationBufferMemory(

    memory_key="chat_history",

    return_messages=True
)

In [20]:
#Add memory to agent. Use agent type conversation_react_description for both memory and reasoning

from langchain.agents import initialize_agent
from langchain.agents import AgentType
from langchain.agents import Tool

agent = initialize_agent(

    tools=tools,

    llm=llm,

    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,

    memory=memory,

    verbose=True,

    max_iterations=3
)

/tmp/ipykernel_81/1047603337.py:7: LangChainDeprecationWarning: LangChain agents will continue to be supported, but it is recommended for new use cases to be built with LangGraph. LangGraph offers a more flexible and full-featured framework for building agents, including support for tool-calling, persistence of state, and human-in-the-loop workflows. For details, refer to the `LangGraph documentation <https://langchain-ai.github.io/langgraph/>`_ as well as guides for `Migrating from AgentExecutor <https://python.langchain.com/docs/how_to/migrate_agent/>`_ and LangGraph's `Pre-built ReAct agent <https://langchain-ai.github.io/langgraph/how-tos/create-react-agent/>`_.
  agent = initialize_agent(


In [21]:
#Change prompt to plan & reason better with memory 

from langchain.prompts import PromptTemplate

planning_prompt = PromptTemplate(

    input_variables=["query"],

    template="""
You are SupportGenie AI.

Before answering:

1. Understand the customer's goal
2. Break the problem into steps
3. Decide which tools are needed
4. Use tools only if necessary
5. Escalate unresolved cases

Customer Query:
{query}

Provide:
- Plan
- Actions
- Final Response
"""
)

In [22]:
#Set session timeout

SESSION_TIMEOUT = 30  # minutes


#session reset 
memory.clear()

In [24]:
#memory aware application 

def run_conversational_agent():

    print("=" * 60)
    print("SupportGenie AI - Conversational Agent")
    print("=" * 60)

    while True:

        query = input("\nCustomer: ")

        if query.lower() == "exit":
            break

        try:

            response = agent.run(query)

            print(f"\nAgent: {response}")

        except Exception as e:

            print(
                "\nAgent: Temporary issue detected."
            )

In [27]:
# =====================================================
# APPLICATION ENTRY POINT
# =====================================================

if __name__ == "__main__":

    run_conversational_agent()

SupportGenie AI - Conversational Agent


> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: OrderStatusTool
Action Input: 1001
Observation: Order 1001 is currently Shipped and expected delivery is 2026-05-20.
Thought:Do I need to use a tool? No  
AI: Your order with ID 1001 is currently shipped and is expected to be delivered on May 20, 2026. If you have any other questions or need further assistance, feel free to ask!

> Finished chain.

Agent: Your order with ID 1001 is currently shipped and is expected to be delivered on May 20, 2026. If you have any other questions or need further assistance, feel free to ask!


> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? No
AI: Your order with ID 1001 is currently shipped and is expected to be delivered on May 20, 2026. If you have any other questions or need further assistance, feel free to ask!
```

> Finished chain.

Agent: Your order with ID 1001 is currently shipped and is e

**Analysis of memory module: **

1. Please refer to the output logs above. The Queries to retreive order id works fine without user reentering the order id as 1001. The agent rememebers the order id for 30 mins. 

2. Also tested with password related issues. Agent remembers which response it has given and when input that the response is not resolving the issue, the agent gives a different response each time.

3. Additionaly, the agent remembers my history of questions when asked what issues I have been facing in the past. 




PHASE 7 -ADAPTIVE BEHAVIOUR

This phase moves the agent from being static to an adaptive agent demonstrating 

1. Continuous improvement,
2. Behaviour adaptation,
3. Learning loops,
4. Personalization,
5. Production grade AI engineering practices.


Objectives are to build :
1. Feedback collection
2. Feedback storage
3. Behaviour adaptation
4. Before vs After comparison

The following is how behaviour changes are interpreted and handled:

| Feedback                 | Behaviour Change                |
| ------------------------ | ------------------------------- |
| “Too verbose”            | Shorter responses               |
| “Wrong tool selected”    | Update routing rules            |
| “Escalate sooner”        | Lower escalation threshold      |
| “Customer prefers email” | Remember preference             |
| “Response inaccurate”    | Strengthen retrieval dependency |


**Architecture:**

Customer Interaction
        ↓
Response Generated
        ↓
Feedback Collection
        ↓
Feedback Store
        ↓
Adaptation Engine
        ↓
Behaviour Rules Updated
        ↓
Future Improved Responses

Creating feedback.json to store customer feedback wiht the following details:

| Field      | Purpose                           |
| ---------- | --------------------------------- |
| query      | Original question                 |
| response   | Agent response                    |
| feedback   | positive/negative                 |
| issue_type | routing, verbosity, hallucination |
| timestamp  | audit trail                       |


In [3]:
#Support function to save feedback into feedback.json

import json
from datetime import datetime

FEEDBACK_FILE = "feedback.json"

def save_feedback(

    query,
    response,
    feedback,
    issue_type
):

    entry = {

        "query": query,

        "response": response,

        "feedback": feedback,

        "issue_type": issue_type,

        "timestamp": str(datetime.now())
    }

    try:

        with open(
            FEEDBACK_FILE,
            "r"
        ) as f:

            data = json.load(f)

    except:

        data = []

    data.append(entry)

    with open(
        FEEDBACK_FILE,
        "w"
    ) as f:

        json.dump(
            data,
            f,
            indent=2
        )

In [4]:
#Support function to gather feedback from user 

feedback = input(

    "\nWas this response helpful? "
    "(yes/no): "
)

if feedback.lower() == "no":

    issue = input(

        "Issue type "
        "(verbosity/routing/inaccurate/etc): "
    )

    save_feedback(

        query=user_query,

        response=response,

        feedback="negative",

        issue_type=issue
    )

else:

    save_feedback(

        query=user_query,

        response=response,

        feedback="positive",

        issue_type="none"
    )

NameError: name 'user_query' is not defined

In [5]:
#Load feedback statistics

def load_feedback_stats():

    try:

        with open(
            FEEDBACK_FILE,
            "r"
        ) as f:

            data = json.load(f)

    except:

        return {}

    stats = {

        "verbosity": 0,
        "routing": 0,
        "inaccurate": 0
    }

    for item in data:

        if item["feedback"] == "negative":

            issue = item["issue_type"]

            if issue in stats:

                stats[issue] += 1

    return stats

In [4]:
#Dynamic prompt builder

def build_dynamic_prompt():

    stats = load_feedback_stats()

    verbosity_instruction = ""

    if stats["verbosity"] >= 3:

        verbosity_instruction = (
            "Keep responses extremely concise."
        )

    else:

        verbosity_instruction = (
            "Provide moderately detailed responses."
        )

    prompt = f"""
You are SupportGenie AI.

RULES:
- Never hallucinate policies
- Escalate sensitive issues
- Use retrieved context only

STYLE:
{verbosity_instruction}
"""

    return prompt

**FINAL IMPLEMENTATION CODE FROM PHASES 3, 4, 5, 6 & 7:**
**=====================================================**


The below code is the final and integrated version. Please run this for testing all phases together.

In [15]:
# ============================================================
# SUPPORTGENIE AI
# PHASE 3 + 4 + 5 + 6 + 7 FULL IMPLEMENTATION
# ============================================================

# FEATURES INCLUDED
# ------------------------------------------------------------
# Phase 3 -> LLM + Prompt Engineering
# Phase 4 -> RAG + Semantic Search
# Phase 5 -> Tool Calling + Guardrails
# Phase 6 -> Memory + Multi-step Conversations
# Phase 7 -> Feedback + Adaptive Behaviour
# ============================================================


# ============================================================
# INSTALL REQUIRED PACKAGES
# ============================================================

# pip install langchain==0.1.20
# pip install langchain-openai==0.1.7
# pip install langchain-community==0.0.38
# pip install faiss-cpu
# pip install python-dotenv
# pip install tiktoken


# ============================================================
# PROJECT STRUCTURE
# ============================================================

"""
support-ai-agent/

│
├── data/
│   ├── refund_policy.txt
│   ├── shipping_policy.txt
│   ├── escalation_policy.txt
│   └── account_security.txt
│
├── logs/
│
├── feedback.json
│
├── .env
│
└── app.py
"""


# ============================================================
# IMPORTS
# ============================================================

import os
import re
import json
import logging

from datetime import datetime

from dotenv import load_dotenv


# ============================================================
# LOAD ENV VARIABLES
# ============================================================

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

#Load langsmith env variables for tracing

os.environ["LANGCHAIN_TRACING_V2"] = "true"

os.environ["LANGCHAIN_API_KEY"] = os.getenv(
    "LANGCHAIN_API_KEY"
)

os.environ["LANGCHAIN_PROJECT"] = (
    "SupportGenie-AI-Agent"
)



# ============================================================
# LANGCHAIN IMPORTS
# ============================================================

from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

from langchain.prompts import PromptTemplate

from langchain.memory import ConversationBufferMemory

from langchain.chains import LLMChain

from langchain.agents import (
    initialize_agent,
    AgentType,
    Tool
)

from langchain_community.vectorstores import FAISS

from langchain_community.document_loaders import TextLoader

from langchain.text_splitter import (
    RecursiveCharacterTextSplitter
)

from langsmith import traceable

# ============================================================
# LOGGING CONFIGURATION
# ============================================================

os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    filename="logs/support_agent.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)


# ============================================================
# FEEDBACK STORAGE FILE
# ============================================================

FEEDBACK_FILE = "feedback.json"

if not os.path.exists(FEEDBACK_FILE):

    with open(FEEDBACK_FILE, "w") as f:

        json.dump([], f)


# ============================================================
# SAFETY KEYWORDS
# ============================================================

UNSAFE_KEYWORDS = [

    "hack",
    "steal account",
    "delete another user",
    "fake refund",
    "bypass payment"
]

SENSITIVE_KEYWORDS = [

    "lawsuit",
    "legal",
    "fraud",
    "harassment",
    "police"
]


# ============================================================
# MOCK DATABASES
# ============================================================

ORDERS_DB = {

    "1001": {
        "status": "Shipped",
        "delivery_date": "2026-05-20"
    },

    "1002": {
        "status": "Processing",
        "delivery_date": "2026-05-25"
    },

    "1003": {
        "status": "Delivered",
        "delivery_date": "2026-05-10"
    }
}

ESCALATION_QUEUE = []


# ============================================================
# INPUT VALIDATION
# ============================================================

def validate_input(query):

    if not query.strip():

        return False, "Input cannot be empty."

    if len(query) > 1000:

        return False, "Input too long."

    return True, None


# ============================================================
# PII SANITIZATION
# ============================================================

def sanitize_text(text):

    text = re.sub(
        r'\S+@\S+',
        '[EMAIL_REDACTED]',
        text
    )

    text = re.sub(
        r'\b\d{10}\b',
        '[PHONE_REDACTED]',
        text
    )

    return text


# ============================================================
# SAFETY CHECK
# ============================================================

def safety_check(query):

    q = query.lower()

    for word in UNSAFE_KEYWORDS:

        if word in q:

            return (
                False,
                "I cannot assist with unsafe or unauthorized requests."
            )

    return True, None


# ============================================================
# FEEDBACK SAFETY CHECK
# ============================================================

def safe_feedback_check(text):

    blocked_words = [

        "hack",
        "bypass",
        "steal"
    ]

    for word in blocked_words:

        if word in text.lower():

            return False

    return True


# ============================================================
# FEEDBACK STORAGE
# ============================================================
@traceable
def save_feedback(

    query,
    response,
    feedback,
    issue_type
):

    if not safe_feedback_check(issue_type):

        return

    entry = {

        "query": query,

        "response": response,

        "feedback": feedback,

        "issue_type": issue_type,

        "timestamp": str(datetime.now())
    }

    with open(FEEDBACK_FILE, "r") as f:

        data = json.load(f)

    data.append(entry)

    with open(FEEDBACK_FILE, "w") as f:

        json.dump(data, f, indent=2)


# ============================================================
# LOAD FEEDBACK STATS
# ============================================================

def load_feedback_stats():

    with open(FEEDBACK_FILE, "r") as f:

        data = json.load(f)

    stats = {

        "verbosity": 0,
        "routing": 0,
        "inaccurate": 0
    }

    for item in data:

        if item["feedback"] == "negative":

            issue = item["issue_type"]

            if issue in stats:

                stats[issue] += 1

    return stats


# ============================================================
# ADAPTIVE PROMPT BUILDER
# ============================================================
@traceable
def build_dynamic_prompt():

    stats = load_feedback_stats()

    verbosity_instruction = ""

    if stats["verbosity"] >= 3:

        verbosity_instruction = (
            "Keep responses short and concise."
        )

    else:

        verbosity_instruction = (
            "Provide moderately detailed responses."
        )

    return f"""
You are SupportGenie AI.

RULES:
- Use retrieved context only
- Give professional answers
- Never hallucinate policies
- Escalate unresolved issues
- Refuse unsafe requests

STYLE:
{verbosity_instruction}
"""


# ============================================================
# LOAD DOCUMENTS
# ============================================================

documents = []

for file in os.listdir("data"):

    if file.endswith(".txt"):

        loader = TextLoader(
            os.path.join("data", file)
        )

        documents.extend(loader.load())


# ============================================================
# TEXT SPLITTING
# ============================================================

text_splitter = RecursiveCharacterTextSplitter(

    chunk_size=70,

    chunk_overlap=20
)

splits = text_splitter.split_documents(documents)


# ============================================================
# EMBEDDINGS + VECTOR DB
# ============================================================

embeddings = OpenAIEmbeddings()

vectordb = FAISS.from_documents(

    splits,
    embeddings
)

retriever = vectordb.as_retriever(

    search_kwargs={"k": 3}
)


# ============================================================
# INITIALIZE LLM
# ============================================================

llm = ChatOpenAI(

    model="gpt-4o-mini",

    temperature=0.2
)


# ============================================================
# MEMORY
# ============================================================

memory = ConversationBufferMemory(

    memory_key="chat_history",

    return_messages=True
)


# ============================================================
# TOOL 1 -> ORDER STATUS
# ============================================================
@traceable
def get_order_status(order_id):

    if order_id in ORDERS_DB:

        order = ORDERS_DB[order_id]

        return (
            f"Order {order_id} is "
            f"{order['status']} and expected "
            f"delivery is "
            f"{order['delivery_date']}."
        )

    return "Order ID not found."


# ============================================================
# TOOL 2 -> ESCALATION
# ============================================================
@traceable
def escalate_ticket(reason):

    ESCALATION_QUEUE.append(reason)

    return (
        "Issue escalated to human support."
    )


# ============================================================
# TOOL 3 -> REFUND ELIGIBILITY
# ============================================================
@traceable
def refund_eligibility(days):

    try:

        days = int(days)

        if days <= 30:

            return (
                "Customer is eligible for refund."
            )

        return (
            "Refund requires managerial approval."
        )

    except:

        return "Invalid refund duration."


# ============================================================
# LANGCHAIN TOOLS
# ============================================================

tools = [

    Tool(

        name="OrderStatusTool",

        func=get_order_status,

        description="""
        ONLY use for:
        - tracking orders
        - shipment status
        - delivery updates

        Input should ONLY be order ID.
        """
    ),

    Tool(

        name="EscalationTool",

        func=escalate_ticket,

        description="""
        Use for:
        - legal threats
        - fraud complaints
        - harassment
        - unresolved issues
        """
    ),

    Tool(

        name="RefundEligibilityTool",

        func=refund_eligibility,

        description="""
        ONLY use for:
        - refund eligibility
        - refund duration
        - return windows

        Input should ONLY be number of days.
        """
    )
]


# ============================================================
# DYNAMIC PROMPT TEMPLATE
# ============================================================

rag_prompt = PromptTemplate(

    input_variables=[
        "context",
        "query",
        "chat_history",
        "system_rules"
    ],

    template="""
{system_rules}

Conversation History:
{chat_history}

Retrieved Context:
{context}

Customer Query:
{query}

Provide:
1. Intent
2. Action Taken
3. Final Response
"""
)


# ============================================================
# RAG CHAIN
# ============================================================

rag_chain = LLMChain(

    llm=llm,

    prompt=rag_prompt
)


# ============================================================
# RAG PIPELINE
# ============================================================
@traceable
def rag_pipeline(user_query):

    retrieved_docs = retriever.get_relevant_documents(
        user_query
    )

    context = "\n\n".join(
        [doc.page_content for doc in retrieved_docs]
    )

    if not context.strip():

        return (
            "No verified information found. "
            "Escalating to support."
        )

    system_rules = build_dynamic_prompt()

    response = rag_chain.run(

        context=context,

        query=user_query,

        chat_history=memory.buffer,

        system_rules=system_rules
    )

    return response


# ============================================================
# AGENT INITIALIZATION
# ============================================================

agent = initialize_agent(

    tools=tools,

    llm=llm,

    agent=AgentType.CONVERSATIONAL_REACT_DESCRIPTION,

    memory=memory,

    verbose=True,

    max_iterations=3,

    handle_parsing_errors=True
)


# ============================================================
# MAIN SUPPORT AGENT
# ============================================================
@traceable
def support_agent(user_query):

    # =====================================================
    # INPUT VALIDATION
    # =====================================================

    valid, error = validate_input(user_query)

    if not valid:

        return error

    # =====================================================
    # SAFETY CHECK
    # =====================================================

    safe, response = safety_check(user_query)

    if not safe:

        return response

    # =====================================================
    # LEGAL / FRAUD ESCALATION
    # =====================================================

    if any(
        word in user_query.lower()
        for word in SENSITIVE_KEYWORDS
    ):

        return agent.run(
            f"Escalate issue: {user_query}"
        )

    # =====================================================
    # ORDER TRACKING
    # =====================================================

    if "order" in user_query.lower():

        numbers = re.findall(r'\d+', user_query)

        if numbers:

            return agent.run(
                f"Track order {numbers[0]}"
            )

    # =====================================================
    # REFUND ROUTING
    # =====================================================

    refund_keywords = [

        "refund",
        "return",
        "money back",
        "cancel order"
    ]

    if any(
        word in user_query.lower()
        for word in refund_keywords
    ):

        numbers = re.findall(r'\d+', user_query)

        if numbers:

            return agent.run(
                f"Check refund eligibility "
                f"for {numbers[0]} days"
            )

    # =====================================================
    # DEFAULT RAG
    # =====================================================

    return rag_pipeline(user_query)


# ============================================================
# LOGGING
# ============================================================
@traceable
def log_interaction(query, response):

    safe_query = sanitize_text(query)

    safe_response = sanitize_text(response)

    logging.info(

        f"QUERY={safe_query} | "
        f"RESPONSE={safe_response}"
    )


# ============================================================
# FEEDBACK SUMMARY
# ============================================================
@traceable
def feedback_summary():

    with open(FEEDBACK_FILE, "r") as f:

        data = json.load(f)

    total = len(data)

    positive = sum(

        1 for x in data

        if x["feedback"] == "positive"
    )

    if total == 0:

        print("No feedback available.")

        return

    print(
        f"\nPositive Feedback Rate: "
        f"{positive / total:.2f}"
    )


# ============================================================
# MAIN APPLICATION LOOP
# ============================================================

def run_application():

    print("=" * 70)
    print("SupportGenie AI - Enterprise AI Support Agent")
    print("=" * 70)

    while True:

        user_query = input("\nCustomer: ")

        # =================================================
        # EXIT CONDITION
        # =================================================

        if user_query.lower() == "exit":

            print("\nSession Ended.")

            break

        try:

            # =============================================
            # GENERATE RESPONSE
            # =============================================

            response = support_agent(user_query)

            print("\nAgent:")
            print(response)

            # =============================================
            # LOGGING
            # =============================================

            log_interaction(
                user_query,
                response
            )

            # =============================================
            # FEEDBACK COLLECTION
            # =============================================

            feedback = input(
                "\nWas this helpful? (yes/no): "
            )

            if feedback.lower() == "no":

                issue = input(
                    "Issue type "
                    "(verbosity/routing/inaccurate): "
                )

                save_feedback(

                    query=user_query,

                    response=response,

                    feedback="negative",

                    issue_type=issue
                )

            else:

                save_feedback(

                    query=user_query,

                    response=response,

                    feedback="positive",

                    issue_type="none"
                )

        except Exception as e:

            logging.error(str(e))

            print(
                "\nAgent: Temporary system issue detected."
            )

    # =====================================================
    # FEEDBACK ANALYTICS
    # =====================================================

    feedback_summary()


# ============================================================
# APPLICATION ENTRY POINT
# ============================================================

if __name__ == "__main__":

    run_application()

SupportGenie AI - Enterprise AI Support Agent


/tmp/ipykernel_80/2953932531.py:615: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retrieved_docs = retriever.get_relevant_documents(
/tmp/ipykernel_80/2953932531.py:632: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = rag_chain.run(



Agent:
1. **Intent**: The customer is inquiring about the status of their order.

2. **Action Taken**: Since the specific order details are not available, I cannot provide the exact status of the order. However, I can inform the customer about the standard delivery times.

3. **Final Response**: Thank you for your inquiry regarding the status of your order. Our standard delivery time is typically 5-7 business days. If your order is being shipped internationally, it may take 10-15 business days. If you believe that your order has exceeded 20 business days, please reach out to our customer service team for further assistance.


> Entering new AgentExecutor chain...
```
Thought: Do I need to use a tool? Yes
Action: OrderStatusTool
Action Input: 1001
Observation: Order 1001 is Shipped and expected delivery is 2026-05-20.
Thought:Do I need to use a tool? No  
AI: Your order 1001 has been shipped and is expected to be delivered on May 20, 2026.

> Finished chain.

Agent:
Your order 1001 has

**Analysis on phase 7 (feedback)**

On this query:

Can i sue you for delay in shipping, observe the response above:

Before feedback:

Agent:
1. **Intent**: The customer is expressing frustration regarding a shipping delay and is inquiring about the possibility of legal action due to this issue.

2. **Action Taken**: I have acknowledged the customer's concern regarding the shipping delay and provided information about the standard delivery time. I have also noted that compensation may be requested if delivery exceeds 20 business days.

3. **Final Response**: While I understand your frustration regarding the shipping delay, legal action is not typically a viable option for such matters. Our standard delivery time is 5-7 business days, and if your order exceeds 20 business days, you may request compensation. If you have further concerns or require additional assistance, please let me know, and I can escalate this issue for you.

After giving feedback as "inaccurate"

Agent:
1. **Intent**: The customer is inquiring about the possibility of legal action due to a delay in shipping.

2. **Action Taken**: I have reviewed the shipping policy, which states that standard delivery time is 5-7 business days. Since the order is expected to be delivered on May 20, 2026, it is significantly beyond the standard delivery timeframe. 

3. **Final Response**: While I understand your frustration regarding the shipping delay, legal action is typically a last resort. Our shipping policy allows for delivery within 5-7 business days, and if delivery exceeds 20 business days, we may be able to assist you further. I recommend reaching out to our customer service team to discuss your concerns and explore possible resolutions. If you have further questions or need assistance, please let me know.


In the second response, agent is escalating to customer service as in the escalation policy while in the first one it just quotes from shipping policy. 

2. If feedback is given as verbose, i see shorter responses for following queries, as in the below query:

Query: Can i use crypto currency for payment?

Before feedback:

Agent:
1. **Intent**: The customer is inquiring about the possibility of using cryptocurrency as a payment method.

2. **Action Taken**: I have reviewed the available information regarding payment methods but do not have specific details about the acceptance of cryptocurrency.

3. **Final Response**: Currently, I do not have information regarding the acceptance of cryptocurrency for payments. I recommend checking with customer service or the payment section of our website for the most accurate and up-to-date information on accepted payment methods.

After verbose feedback:

Agent:
1. **Intent**: The customer is inquiring about the possibility of using cryptocurrency for transactions.

2. **Action Taken**: I have reviewed the available information regarding payment methods, but there is no specific policy mentioned about the acceptance of cryptocurrency.

3. **Final Response**: Currently, I do not have information regarding the acceptance of cryptocurrency for transactions. I recommend checking with customer service for the most accurate and up-to-date information on payment options.


Final response is a bit shorter. 

**PHASE 9 - DEPLOYMENT READINESS & OBSERVABILITY**

Tech Stack:

Langsmith - observability (tracing & logs)
Github - Deployment 





In [4]:
pip install langsmith

Defaulting to user installation because normal site-packages is not writeableRequirement already satisfied: langsmith in /usr/local/lib/python3.10/site-packages (0.4.14)Requirement already satisfied: httpx<1,>=0.23.0 in /usr/local/lib/python3.10/site-packages (from langsmith) (0.28.1)Requirement already satisfied: orjson>=3.9.14 in /usr/local/lib/python3.10/site-packages (from langsmith) (3.11.2)Requirement already satisfied: packaging>=23.2 in /usr/local/lib/python3.10/site-packages (from langsmith) (23.2)Requirement already satisfied: pydantic<3,>=1 in /usr/local/lib/python3.10/site-packages (from langsmith) (2.11.7)Requirement already satisfied: requests-toolbelt>=1.0.0 in /usr/local/lib/python3.10/site-packages (from langsmith) (1.0.0)Requirement already satisfied: requests>=2.0.0 in /usr/local/lib/python3.10/site-packages (from langsmith) (2.32.5)Requirement already satisfied: zstandard>=0.23.0 in /usr/local/lib/python3.10/site-packages (from langsmith) (0.24.0)Requirement already

In [12]:
#Setting up langsmith env variables 

import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"

os.environ["LANGCHAIN_API_KEY"] = os.getenv(
    "LANGCHAIN_API_KEY"
)

os.environ["LANGCHAIN_PROJECT"] = (
    "SupportGenie-AI-Agent"
)


#To add custom tracing

from langsmith import traceable

Analysis of Langsmith Observability:

1. Added traceability to the critical functions invloving orchestration, routing, tools, feedback adaptation 

@traceable
support_agent()

@traceable
rag_pipeline()

@traceable
get_order_status()

@traceable
refund_eligibility()

@traceable
escalate_ticket()

@traceable
save_feedback()

2. I'm seeing the logs in langsmith portal. Screenshots attached in the evaluation report. 

**DEPLOYMENT**

